# PromptEngineering Lab

## Unidad 2 - Clase 3: IA Generativa y Prompt Engineering

Laboratorio practico con **modelos en español** (misma configuracion que Clase 02):

- Generacion: `datificate/gpt2-small-spanish`
- Sentimiento: `pysentimiento/robertuito-sentiment-analysis`

Tecnicas: anatomia del prompt, zero/one/few-shot, CoT, role y context prompting.

> Ejecuta **Run All**. Si cambias archivos en `src/`, reinicia kernel.

## Objetivos
1. Construir prompts estructurados en español.
2. Aplicar 6 tecnicas de prompting con GPT-2 Spanish.
3. Comparar zero-shot vs modelo de sentimiento Robertuito.
4. Resolver el caso TechNova SaaS.

## Nota sobre los modelos

Usamos los mismos modelos que en la Clase 02 (carpeta CLASE). GPT-2 Spanish fue entrenado para **continuar texto** en español. El objetivo es dominar las **tecnicas de prompting**, no igualar ChatGPT.

In [1]:
%matplotlib inline

import importlib
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from transformers import pipeline

warnings.filterwarnings('ignore', category=UserWarning)

def find_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / 'src').exists() and 'PromptEngineering' in folder.name:
            return folder
    raise FileNotFoundError('Abre el notebook desde PromptEngineering_Lab/notebooks/')

ROOT = find_root(Path.cwd())
SRC = ROOT / 'src'
IMAGES = ROOT / 'images'
IMAGES.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import prompt_builder
import strategies
import llm_client
import evaluator
import utils

importlib.reload(prompt_builder)
importlib.reload(strategies)
importlib.reload(llm_client)
importlib.reload(evaluator)
importlib.reload(utils)

from prompt_builder import build_structured_prompt, STANDARD_CONSTRAINTS
from strategies import zero_shot, one_shot, few_shot_from_dataframe, chain_of_thought, role_prompt, context_prompt
from llm_client import LLMClient, LLMConfig, MODEL_GENERACION, MODEL_SENTIMIENTO
from evaluator import score_response, compare_strategies
from utils import load_dataset

print('Entorno listo. ROOT =', ROOT)

Entorno listo. ROOT = c:\Users\juand\GitHub\M3-Cientifico-Datos-IA-Aplicada-DevSeniorCode\02_nlp_ia_generativa\clase_03_ia_generativa_prompt_engineering\PromptEngineering_Lab


# Parte 1 — Cargar modelos en español

In [2]:
config = LLMConfig(model_name=MODEL_GENERACION, max_new_tokens=50, temperature=0.7, seed=42)
client = LLMClient(config=config)
print('Cargando GPT-2 en español...')
client.load()
print('Cargando Robertuito (sentimiento)...')
sentimientos = pipeline('sentiment-analysis', model='pysentimiento/robertuito-sentiment-analysis')
print('Modelos listos.')

Cargando GPT-2 en español...


Device set to use cpu


Cargando Robertuito (sentimiento)...


Device set to use cpu


Modelos listos.


# Parte 2 — Anatomia del prompt en español

In [3]:
prompt = build_structured_prompt(
    role='Eres un analista de soporte al cliente.',
    instruction='Clasifica el sentimiento como positivo, negativo o neutral.',
    constraints='Responde SOLO con la etiqueta. Sin explicacion.',
    output_format='Una sola palabra',
    user_input='El producto llegó roto y nadie respondió a mis correos.',
)
print(prompt)
print('\n--- Respuesta ---')
print(client.generate_completion(prompt, max_new_tokens=10, temperature=0.3))

Eres un analista de soporte al cliente.

Tarea: Clasifica el sentimiento como positivo, negativo o neutral.

Reglas: Responde SOLO con la etiqueta. Sin explicacion.

Formato de salida: Una sola palabra

Entrada: El producto llegó roto y nadie respondió a mis correos.

Respuesta:

--- Respuesta ---
Repetiré en el producto.


# Parte 3 — Zero-shot, one-shot y few-shot

In [4]:
df_tickets = load_dataset('tickets_soporte.csv', ROOT)
display(df_tickets.head())

ticket_test = 'El sistema de pagos está caído no puedo procesar reembolsos'

p_zero = zero_shot('Clasifica urgencia: alta, media, baja. Solo etiqueta:', ticket_test)
p_one = one_shot('Clasifica urgencia.', 'Sistema caido', 'alta', ticket_test)
p_few = few_shot_from_dataframe(
    'Clasifica urgencia: alta, media, baja. Solo etiqueta:',
    df_tickets, 'ticket', 'urgencia', ticket_test, n_examples=3,
)

comparacion = []
for nombre, p in [('zero-shot', p_zero), ('one-shot', p_one), ('few-shot', p_few)]:
    resp = client.generate_completion(p, max_new_tokens=5, temperature=0.3)
    comparacion.append({'estrategia': nombre, 'respuesta': resp})
display(pd.DataFrame(comparacion))

,ticket,urgencia,categoria
0,Sistema caido no puedo facturar a ningun cliente,alta,tecnico
1,Como cambio mi contraseña del portal,baja,cuenta
2,Error intermitente en reportes de ventas,media,tecnico
3,No recibi mi pedido despues de 2 semanas,alta,logistica
4,Quiero saber los horarios de atencion,baja,informacion


,estrategia,respuesta
0,zero-shot,El sistema de pagos está
1,one-shot,baja\nEntrada:
2,few-shot,baja\n\nEntrada


# Parte 4 — Chain-of-Thought y role prompting

In [5]:
problema = 'Una tienda tiene 23 manzanas. Usan 20 para pasteles y compran 6 más. ¿Cuántas manzanas tienen?'
p_cot = chain_of_thought(problema)
print('=== CoT ===')
print(client.generate_completion(p_cot, max_new_tokens=60, temperature=0.5))

roles = [
    ('un profesor paciente para principiantes', '¿Qué es el aprendizaje automático?'),
    ('un abogado laboral', '¿Qué es una cláusula de no competencia?'),
]
for rol, q in roles:
    p = role_prompt(rol, q)
    print(f'\n=== Rol: {rol[:30]}... ===')
    print(client.generate_completion(p, max_new_tokens=40, temperature=0.7))

=== CoT ===
El juego es un juego de palabras, que se juega en una pantalla de 2x2 píxeles (20×40) con un tamaño de 8x8 píxeles.

Es una aplicación libre de software gratuito que permite a los usuarios jugar con otros juegos y crear nuevas experiencias.

=== Rol: un profesor paciente para prin... ===
El aprendizaje automático se refiere a la aplicación de una estrategia para lograr objetivos, y su uso debe ser dirigido en la dirección correcta (y más bien en términos de los objetivos que se ven afectados). La

=== Rol: un abogado laboral... ===
La cláusula de no competencia es una cláusula que se aplica cuando un grupo de personas son capaces de participar en una actividad económica o social, y tienen habilidades especiales de decisión de negocios o de administración empresarial para


# Parte 5 — Context prompting

In [6]:
df_docs = load_dataset('documentos_empresa.csv', ROOT)
doc = df_docs[df_docs['titulo'] == 'Politica de devoluciones'].iloc[0]

preguntas = [
    '¿Puedo devolver un producto después de 25 días?',
    '¿Cuál es el salario del CEO?',
]
for q in preguntas:
    p = context_prompt(doc['contenido'], q, safe=True)
    resp = client.generate_completion(p, max_new_tokens=30, temperature=0.3)
    print(f'P: {q}')
    print(f'R: {resp}')
    print('-' * 50)

P: ¿Puedo devolver un producto después de 25 días?
R: No.

La respuesta es:

El producto está sujeto a revisión por la Comisión Europea para su adopción final.

En caso de
--------------------------------------------------
P: ¿Cuál es el salario del CEO?
R: No.

El producto debe estar sin uso y con factura original. No se aceptan productos personalizados. El reembolso se procesa en 5
--------------------------------------------------


# Parte 6 — Comparar prompting vs modelo de sentimiento

In [7]:
df_reviews = load_dataset('reviews_sentiment.csv', ROOT)
fila = df_reviews.iloc[0]
texto = fila['texto']

p_zs = zero_shot('Clasifica sentimiento positivo/negativo. Solo etiqueta:', texto)
r_prompt = client.generate_completion(p_zs, max_new_tokens=5, temperature=0.3)
r_modelo = sentimientos(texto)[0]

display(pd.DataFrame([{
    'texto': texto[:50],
    'sentimiento_real': fila['sentimiento'],
    'zero_shot_gpt': r_prompt,
    'robertuito': r_modelo['label'],
    'score': round(r_modelo['score'], 3),
}]))

resultados = []
for nombre, p in [('zero-shot', p_zero), ('few-shot', p_few), ('cot', p_cot)]:
    resp = client.generate_completion(p, max_new_tokens=30, temperature=0.3)
    resultados.append({'estrategia': nombre, 'respuesta': resp, 'keywords': ['alta', 'media', 'baja']})
evaluados = compare_strategies(resultados)
df_eval = pd.DataFrame(evaluados)
display(df_eval[['estrategia', 'has_content', 'total']])

,texto,sentimiento_real,zero_shot_gpt,robertuito,score
0,"Excelente producto, llegó rápido y funciona pe...",positivo,"Excelente producto, se",POS,0.976


,estrategia,has_content,total
0,zero-shot,1,3
1,few-shot,1,3
2,cot,1,3


# Parte 7 — Caso TechNova SaaS

In [8]:
def clasificar(ticket):
    p = few_shot_from_dataframe(
        'Clasifica urgencia: alta, media, baja. Solo etiqueta:',
        df_tickets, 'ticket', 'urgencia', ticket, n_examples=4,
    )
    return client.generate_completion(p, max_new_tokens=5, temperature=0.3)

def responder(pregunta):
    ctx = '\n'.join(f"{r['titulo']}: {r['contenido']}" for _, r in df_docs.iterrows())
    p = build_structured_prompt(
        role='Eres asistente de soporte de TechNova.',
        context=ctx,
        instruction='Responde la pregunta del cliente.',
        constraints=STANDARD_CONSTRAINTS,
        output_format='Maximo 2 oraciones',
        user_input=pregunta,
    )
    return client.generate_completion(p, max_new_tokens=40, temperature=0.3)

demo = pd.DataFrame([
    {'funcion': 'clasificar', 'entrada': 'La app se cierra al abrir PDFs', 'salida': clasificar('La app se cierra al abrir PDFs')},
    {'funcion': 'responder', 'entrada': '¿Cuál es la política de devoluciones?', 'salida': responder('¿Cuál es la política de devoluciones?')},
    {'funcion': 'clasificar', 'entrada': '¿Cómo cambio mi contraseña?', 'salida': clasificar('¿Cómo cambio mi contraseña?')},
])
display(demo)

,funcion,entrada,salida
0,clasificar,La app se cierra al abrir PDFs,baja\n\nEntrada
1,responder,¿Cuál es la política de devoluciones?,¿Cuál es la política de devoluciones?\n\nEl cl...
2,clasificar,¿Cómo cambio mi contraseña?,baja\n\nEntrada


## Entregables
1. Notebook ejecutado completo.
2. Capturas en `images/` (minimo 4).
3. Reflexion (250-350 palabras): estrategia que usarias en tu trabajo.

## Reflexion
*Escribe aqui tu respuesta sobre que estrategia de prompting usarias en tu trabajo y por que.*